# Clayton Metang — Expedition
High-level workflow using `expedition.py`. Each cell calls one stage; `x.save()` persists config after each step.

In [2]:
%load_ext autoreload
%autoreload 2
import logging
import claytonlib as clayton
from claytonlib.expedition import expedition

# --- Logging ---
# INFO shows per-write-cycle timing; DEBUG adds per-turn RNG details
logging.basicConfig(level=logging.INFO)
# logging.getLogger('claytonlib').setLevel(logging.INFO)

x = expedition("metang")
x.reload()
x.print()
x.chart_options.evaluation_frames_per_write_cycle = 5

[expedition] Reloaded from data/expeditions/metang.json
=== Expedition: metang ===
  pokemon                  metang
  key_seed                 0x0C0E02C2
  setup_delay_s            180
  max_target_s             600
  strategy                 six-bait-then-balls
  criteria                 machete-50-turns-after-5-balls
  eval_strategy            sliding_window_13
  fps_model                linear
  window                   120
  target_delay             28721
  initial_time             2000-07-24T14:45:55
  target_seeds             ['0x1C1562D2']
  metronome_histsz         10
  metronome_second_window  2
  compass_m_delay          2441
  ---
  delay_from_key           28015 frames  (466.92s)


In [13]:
x.adjust(max_target_seconds=400) # Return to 600 after calibration is complete
x.adjust(criteria_name="5-balls-no-flee") # return to machete-50-turns-after-5-balls after calibration is complete
x.save()

[expedition] max_target_seconds = 400
[expedition] criteria_name = '5-balls-no-flee'
[expedition] Saved to data/expeditions/metang.json


# Chart - Finding a target

## Create chart

In [14]:
x.precompute_chart()
x.save()

[expedition] 17:49:21  === precompute_chart ===  (2026-09-13)
[expedition] 17:49:21  charting metang key_seed=0x0C0E02C2 delay=180-400s strategy=six-bait-then-balls criteria=5-balls-no-flee  fps_models=['linear', 'quad'] (union)  workers=12
[expedition] 17:49:22  mdmsh 1/238  elapsed 0.0m  eta ~3.6m
[expedition] 17:49:22  mdmsh 5/238  elapsed 0.0m  eta ~0.7m
[expedition] 17:49:22  mdmsh 10/238  elapsed 0.0m  eta ~0.4m
[expedition] 17:49:22  mdmsh 15/238  elapsed 0.0m  eta ~0.3m
[expedition] 17:49:22  mdmsh 20/238  elapsed 0.0m  eta ~0.2m
[expedition] 17:49:22  mdmsh 25/238  elapsed 0.0m  eta ~0.2m
[expedition] 17:49:22  mdmsh 30/238  elapsed 0.0m  eta ~0.1m
[expedition] 17:49:22  mdmsh 35/238  elapsed 0.0m  eta ~0.1m
[expedition] 17:49:22  mdmsh 40/238  elapsed 0.0m  eta ~0.1m
[expedition] 17:49:22  mdmsh 45/238  elapsed 0.0m  eta ~0.1m
[expedition] 17:49:22  mdmsh 50/238  elapsed 0.0m  eta ~0.1m
[expedition] 17:49:22  mdmsh 55/238  elapsed 0.0m  eta ~0.1m
[expedition] 17:49:22  mdmsh 

## Evaluate chart to find targets

`chart_report()` ranks the best **(boot time, commanded countdown M)** pairs across all candidate boot times (mode A), or the best M for a boot time you pass as `initial_time=` (mode B). It saves its ranked findings so `select_target()` can use them.

In [15]:
x.chart_report()
x.save()

[expedition] 17:49:29  === chart_report ===
Best (boot time, M) pairs  [top 10 of 6]  (jitter kernel, k=3.5):
   #            boot time     M (ms)  target F_b  second  P(capture)   sigma
   1  2000-06-30 14:58:30     202818       12133     208       37.4%    52.7
   2  2000-05-30 14:59:59     222779       13329     228       37.2%    55.2
   3  2000-01-01 14:00:11     199680       11945     205       36.5%    52.3
   4  2000-05-31 14:54:59     300220       17969     305       36.1%    64.1
   5  2000-06-30 14:53:35     199614       11941     205       36.0%    52.3
   6  2000-06-26 14:53:59     389878       23341     395       36.0%    73.0
[expedition] 17:49:43  best target for each of 2464 starting times also saved
[expedition] 17:49:43  findings saved -> data/metang_0C0E02C2/chart_six-bait-then-balls_5-balls-no-flee/chart_report.json  (top 6 + 2464 per-starting-time)
[expedition] Saved to data/expeditions/metang.json


## Choose Target

`select_target()` reads the findings `chart_report()` saved and lets you pick one. It records the chosen **boot time** (`initial_time`), **timer countdown** (`target_timer_delay` = M), and **expected battle frame** (`target_delay` = F_b) on the expedition, then saves.

In [9]:
x.select_target()

[expedition] 16:04:58  === select_target ===



Select by  [t] top ranking   [s] specific starting time   [l] reuse last (07-24 14:45:55)  (blank to cancel):  l


  -> best target for 2000-07-24 14:45:55: M=249817 ms, F_b=14949, P~24.7%
[expedition] Saved to data/expeditions/metang.json
[expedition] 16:04:59  target set: boot 2000-07-24T14:45:55, timer M=249817 ms, expected F_b=14949 (P~24.7%). Saved.
[expedition] 16:04:59  predicted battle time (m/d h:m:s): 07-24 14:50:10  (= boot + 255s; year is the chart's 2000)


{'rank': 2293,
 'initial_time': '2000-07-24T14:45:55',
 'M': 249817,
 'target_delay': 14949,
 'second': 255,
 'p': 0.24677710430965377,
 'sigma': 58.458315599567364,
 'mdmsh': [228, 14]}

## Examine target area

In [11]:
 x.check().chart_check_target_landing()


chart_check_target_landing  (jitter kernel, k=3.5)
boot=2000-07-24T14:45:55  timer M=249817 ms  ->  mean F_b=14949.0 (target_delay=14949)  sigma=58.5
RTC-second distribution (σ_S=0.50s):  255=68%  256=19%  254=13%  257=0%

=== second 255  P(S=255)=67.6%   mdmsh(m,h)=(228, 14)   battle 07-24 14:50:10
    cp(this second) = 25.43%   ->  contributes P·cp = 17.18% to the total
      frame      Δ        seed  hit    weight        w%      cumP%
    --------------------------------------------------------------
      14744   -205  0xE40E3998    ✗    0.0021    0.001%     0.000%
      14745   -204  0xE40E3999    ✗    0.0023    0.002%     0.000%
      14746   -203  0xE40E399A    ✓    0.0024    0.002%     0.002%
      14747   -202  0xE40E399B    ✗    0.0026    0.002%     0.002%
      14748   -201  0xE40E399C    ✗    0.0027    0.002%     0.002%
      14749   -200  0xE40E399D    ✓    0.0029    0.002%     0.004%
      14750   -199  0xE40E399E    ✓    0.0030    0.002%     0.006%
      14751   -198  0

{'p': 0.24677666988941602,
 'seconds': [{'second': 255,
   'p_second': 0.6755839709431751,
   'cp': 0.2542698666841292},
  {'second': 256, 'p_second': 0.1943417458016523, 'cp': 0.22274974823563945},
  {'second': 254, 'p_second': 0.12783129301475996, 'cp': 0.24430308656019642},
  {'second': 257,
   'p_second': 0.0022429902404126826,
   'cp': 0.21260425644700962}],
 'mismatches': None}

# Compass - Identify target

## Calibrate using metronome

In [ ]:
x.metronome_compass()
x.save()

## Finding what seed you hit in safari

`compass_safari()` builds candidates from the calibrated model: for the commanded countdown **M** (set by `select_target`) it sweeps the battle-frame window **F\* ± kσ** across second offsets **δ∈{−1,0,+1}** (off-by-one timer-start timing — each δ uses the *same* frame window). No hand-set delay window.

As you enter observed turns it ranks survivors by **posterior landing probability** (`P(land)`), shows the most-likely seed and which **δ** you hit ("timer on time / +1s late"), and flags when one candidate passes the confidence threshold. Extra commands:

- **`w`** — widen the frame (`k`) and/or second (`±K`) window and re-apply your path so far (also offered automatically on a no-match).
- The set is bounded to the seeds carrying `mass_cap` (default 0.999) of the landing probability; the Jane offload tip triggers on the *prior-weighted* effective count.

Pass `second_offsets=` / `mass_cap=` to override. Afterwards, `x.save_safari_run()` logs the identified seed, observed path, and inferred timer offset to `data/safari_runs.jsonl` (no capture required) for future model retuning.

In [ ]:
x.compass_safari()
x.save()

In [ ]:
# Loop-back: log this run (seed, observed path, inferred timer offset) for model retuning.
# No capture required — records even a fled/ambiguous run.
x.save_safari_run()

# Machete - Finding a path through seed

This is usually triggered during the "Finding what seed you hit in safari" step, but here's some manual activation anyways

## Finding a path for a single seed

In [ ]:
x.machete_one(max_turns=1000)
x.save()